---
title: Refresh backend properties with real-time benchmarking
description: Refresh QPU properties with real-time characterization, use them to select physical qubits, and compare circuit performance.
---

{/* cspell:ignore CPMG dephase ylim xslow */}

# Refresh backend properties with real-time benchmarking
*Usage estimate: 4 minutes on ibm_kingston, including the optional examples (NOTE: This is an estimate only. Your runtime might vary.)*

## Learning outcomes

- Why the properties reported by a QPU can lag behind the device's current behavior, and when it is worth re-measuring them yourself
- What each of the five standard characterization experiments (readout, single-qubit randomized benchmarking, layered two-qubit randomized benchmarking, $T_1$, and $T_2$) measures
- How to run the full characterization suite in a single call or step by step, and get back a backend object with refreshed properties
- How to compare the measured properties against the reported ones, and why some differences reflect measurement methodology rather than device drift

- How to use refreshed properties for qubit selection and compare measured output distributions against those obtained with reported properties

## Prerequisites

- The [Qiskit patterns](/docs/guides/intro-to-patterns) workflow
- [Qiskit Runtime execution modes](/docs/guides/execution-modes), in particular batch mode
- How to explore [QPU information](/docs/guides/qpu-information), such as backend properties and calibration data

## Background

Every IBM Quantum&reg; QPU reports a set of properties that describe how well each of its qubits is currently working: relaxation times ($T_1$), coherence times ($T_2$), readout errors, and one- and two-qubit gate errors. These numbers matter in practice. The transpiler uses them to decide which physical qubits your circuit should run on, error mitigation techniques rely on them, and you might use them to judge whether a device is working well enough for an experiment.

The reported properties come from calibration procedures that typically run about once a day. Superconducting qubits, however, can drift on shorter timescales, and transient defects (so-called two-level systems) can temporarily degrade a qubit that looked excellent at calibration time. In addition, a job is often transpiled well before it actually executes, so decisions based on the reported properties can rest on stale information.

This tutorial uses the `BackendCharacterization` utility from [Qiskit Device Benchmarking](https://github.com/qiskit-community/qiskit-device-benchmarking), which packages the experiment construction, job submission, and curve fitting into a few calls. You spend roughly 70 seconds of QPU time and get back a backend object whose properties reflect the device *right now*, ready to be compared against the reported values or passed on to downstream tools. The final example uses these refreshed values to select physical qubits, executes circuits compiled with both sets of properties, and compares their output distributions.

### The characterization experiments

The suite measures five properties, each with a standard experiment:

- **Readout error** (`readout`): Each qubit is prepared in $|0\rangle$ or $|1\rangle$ and immediately measured. The probability of reading out the wrong state gives the state preparation and measurement (SPAM) error.
- **Single-qubit gate error** (`rb_1q`): Randomized benchmarking (RB) runs sequences of random single-qubit Clifford gates that ideally compose to the identity. The way the survival probability decays with sequence length yields the average error per gate, independent of SPAM errors.
- **Two-qubit gate error** (`rb_2q`): The same RB idea applied to full layers of non-overlapping two-qubit gates executed simultaneously, as in a [layer fidelity](https://arxiv.org/abs/2311.05933) experiment. Because many gates run at once, the resulting error rates include crosstalk effects and are closer to what a deep circuit actually experiences.
- **$T_1$** (`t1`): Each qubit is excited and measured after increasing delays. The decay of the excited-state population gives the relaxation time.
- **$T_2$** (`t2`): A Hahn echo experiment puts each qubit in a superposition, applies an echo pulse halfway through a delay, and measures how quickly phase coherence is lost. By default a single echo is used; a multi-echo (CPMG-style) variant is demonstrated later in this tutorial.

### What refreshing does and does not do

Keep three points in mind when using this workflow:

- The refreshed properties live only in the backend object returned to you. Nothing changes on the device itself or in the calibration data that IBM Quantum reports to other users.
- The measured values are a snapshot taken with a particular methodology. Some of them, notably the layered two-qubit errors and the parallel single-echo $T_2$ values, are measured differently from the reported calibration data, so part of any gap you observe is methodological rather than drift. This tutorial points out where that happens.
- The characterization itself costs QPU time (about 70 seconds for the full suite on a Heron r2 processor), which is the price of up-to-date information.

The workflow follows the four steps of a Qiskit pattern:

- **Step 1: Map classical inputs to a quantum problem.** Choose the device and the set of properties to measure.
- **Step 2: Optimize problem for quantum hardware execution.** The utility builds the experiment circuits directly in the device's native gates and parallelizes them across the chip.
- **Step 3: Execute using Qiskit primitives.** The experiments run as Sampler jobs inside a batch.
- **Step 4: Post-process and return result in desired classical format.** Fit the measurement data into per-qubit error maps, refresh the backend, and compare the measured properties against the reported ones.

## Requirements

Before starting this tutorial, be sure you have the following installed:

- Qiskit SDK v2.0 or later, with [visualization](/docs/api/qiskit/visualization) support
- Qiskit Runtime v0.40 or later (`pip install qiskit-ibm-runtime`)
- Qiskit Device Benchmarking, which also installs [Qiskit Experiments](https://qiskit-community.github.io/qiskit-experiments/) (`pip install git+https://github.com/qiskit-community/qiskit-device-benchmarking.git`)

## Setup

Import the required libraries.

In [1]:
from copy import deepcopy

import logging
import sys

import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import hellinger_fidelity
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Batch, QiskitRuntimeService, SamplerV2

from qiskit_device_benchmarking.utilities.characterization_utils import (
    BackendCharacterization,
    plot_characterization_comparison,
)

The characterization suite builds several experiments, submits multiple jobs, and fits the results, which can take a couple of minutes in total. Enable INFO-level logging so that each stage prints its progress as it happens.

In [2]:
logger = logging.getLogger("qiskit_device_benchmarking")
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler(stream=sys.stdout))

## Small-scale simulator example

This tutorial does not include a simulator example. The workflow measures the physical imperfections of a specific device: how quickly its qubits relax and dephase, and how often its gates and readout fail. An ideal simulator has none of these imperfections, so there is nothing to characterize. You could attach a synthetic noise model to a fake backend, but the experiments would then only recover the numbers you put in yourself. For that reason, we proceed directly to hardware, broken into the four steps of a Qiskit pattern.

## Large-scale hardware example

### Step 1: Map classical inputs to a quantum problem

In this workflow the "problem" is the device itself: the classical inputs are the QPU you want to characterize and the list of properties to measure, and the quantum experiments are the characterization circuits generated from them.

First, select the least busy available QPU with at least 120 qubits. The results shown here were collected on `ibm_kingston`, a Heron r2 processor, on September 16, 2026.

In [3]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=120,
)

Next, choose which properties to measure. You can pass any subset of the following experiments:

- `readout`: SPAM (state preparation and measurement) experiment
- `rb_1q`: isolated single-qubit randomized benchmarking
- `rb_2q`: simultaneous two-qubit randomized benchmarking from a layer fidelity experiment
- `t2`: $T_2$ Hahn echo experiment (a single echo by default)
- `t1`: $T_1$ relaxation experiment

Run the full suite to get a complete picture of the device; drop entries from the list if you only care about some properties and want to save QPU time.

In [4]:
experiments = ["readout", "rb_1q", "rb_2q", "t1", "t2"]

### Step 2: Optimize problem for quantum hardware execution

In most tutorials this is where you would transpile your circuits. Here, `BackendCharacterization` takes care of that internally. It builds the experiment circuits directly in the device's native gates, runs the single-qubit experiments on all qubits in parallel, and schedules the two-qubit benchmarking over disjoint layers of the coupling map so that the entire device is covered with a handful of jobs. Because of this parallelization, the whole suite consumes only about 70 seconds of QPU time.

Initialize the class that manages the workflow:

In [5]:
# Initialize the class used to run the experiments
characterizer = BackendCharacterization(backend)

### Step 3: Execute using Qiskit primitives

The `run_experiments` method builds the circuits and submits them as [Sampler](/docs/guides/get-started-with-sampler) jobs. Run it inside a [`Batch`](/docs/guides/run-jobs-batch) context so that the jobs are scheduled together on the QPU, and the method returns once all jobs have been submitted. (You can also run it outside a batch, or inside a `Session`.)

<Admonition type="note">
You might see a warning about a backend being passed while a session context manager is open. This is expected and can be safely ignored. Layer fidelity may also warn that it does not use the optional `xslow` gate; the experiment uses supported standard single-qubit gates.
</Admonition>

In [6]:
# Run the characterization experiments inside a Batch
with Batch(backend=backend):
    jobs = characterizer.run_experiments(experiments=experiments)

base_primitive.get_mode_service_backend:WARNING:2026-09-16 15:10:33,379: A backend was passed in as the mode but a session context manager is open so this job will run inside this session/batch instead of in job mode.


Building readout experiments


Building 1Q RB experiments


Building 2Q RB experiments


Building T1 experiments


...

Layered two-qubit RB submitted: ['dalekc8pqrnc739622ug', 'dalekhf8gn2s739juam0', 'dalel982fm4c73f1gv9g', 'daleld02fm4c73f1gve0']


Readout job submitted: daleld8pqrnc7396245g


T1 experiment submitted: ['daleldopqrnc7396246g']


T2 (Hahn) experiment submitted: ['dalelef8gn2s739jubjg']


Single-qubit RB submitted: ['dalem6o2fm4c73f1h0c0', 'dalembtr85ps73fbbn00']


The method returns the submitted jobs as a dictionary keyed by experiment, which is useful for tracking them on the [IBM Quantum Platform dashboard](/docs/guides/monitor-job) or for debugging.

In [7]:
# Print all the job IDs for debugging purposes
jobs

{'rb_2q': [<RuntimeJobV2('dalekc8pqrnc739622ug', 'sampler')>,
  <RuntimeJobV2('dalekhf8gn2s739juam0', 'sampler')>,
  <RuntimeJobV2('dalel982fm4c73f1gv9g', 'sampler')>,
  <RuntimeJobV2('daleld02fm4c73f1gve0', 'sampler')>],
 'readout': [<RuntimeJobV2('daleld8pqrnc7396245g', 'sampler')>],
 't1': [<RuntimeJobV2('daleldopqrnc7396246g', 'sampler')>],
 't2': [<RuntimeJobV2('dalelef8gn2s739jubjg', 'sampler')>],
 'rb_1q': [<RuntimeJobV2('dalem6o2fm4c73f1h0c0', 'sampler')>,
  <RuntimeJobV2('dalembtr85ps73fbbn00', 'sampler')>]}

### Step 4: Post-process and return result in desired classical format

#### Analyze the results

The `analyze_results` method waits for the jobs to finish and fits the measurement data: exponential decays for $T_1$ and $T_2$, survival-probability decays for the RB experiments, and assignment matrices for readout. It returns a dictionary of *error maps*, one entry per measured property, each mapping a qubit (or qubit pair) to its measured value.

In [8]:
error_maps = characterizer.analyze_results()

In [9]:
print(f"The following error maps are available: {list(error_maps.keys())}")
readout_q0 = error_maps["readout_error"][0]
print(f"For example, readout error for qubit 0 is {readout_q0}")

The following error maps are available: ['readout_error', 'oneq_error_x', 'oneq_error_sx', 'lf_error_map', 't1_map', 't2_map']
For example, readout error for qubit 0 is 0.010000000000000009


The maps cover readout error, the single-qubit `x` and `sx` gate errors, the layered two-qubit errors (`lf_error_map`, from the layer fidelity experiment), and the $T_1$ and $T_2$ times in seconds.

#### Update the backend properties

The `update_backend` method writes the measured values into the properties of a copy of the backend and returns it. The original `backend` object keeps the reported calibration data, so that we can compare the two, as shown below. Remember that this update is purely local to your Python session; it does not change anything on the device or for other users.

In [10]:
backend_updated = characterizer.update_backend()

Updating readout error


Updating single-qubit X error


Updating single-qubit SX error


Updating two-qubit error


Updating T1


Updating T2


#### Compare the measured properties against the reported ones

Finally, plot the measured real-time properties against the values the backend reports. In each panel the qubits (or qubit pairs) are sorted by the measured value, so the measured curve is smooth by construction and the scatter of the reported values around it shows where the two disagree. A single plot is shown for single-qubit RB because the same measured error per gate is assigned to both the `x` and `sx` gates. Axis limits are chosen from the bulk of the data so that a few extreme outliers do not compress the plots; any points outside the range are counted in an annotation on the plot (or pass `ylim` to override the limits).

In [11]:
plot_characterization_comparison(
    old_props=backend.properties().to_dict(),
    new_props=backend_updated.properties().to_dict(),
    plots=["readout", "rb_1q", "rb_2q", "t1", "t2"],
)

<Image src="/docs/images/tutorials/refresh-backend-properties-with-real-time-benchmarking/extracted-outputs/cell-25-0.svg" alt="Output of the previous code cell" />

<Image src="/docs/images/tutorials/refresh-backend-properties-with-real-time-benchmarking/extracted-outputs/cell-25-1.svg" alt="Output of the previous code cell" />

<Image src="/docs/images/tutorials/refresh-backend-properties-with-real-time-benchmarking/extracted-outputs/cell-25-2.svg" alt="Output of the previous code cell" />

<Image src="/docs/images/tutorials/refresh-backend-properties-with-real-time-benchmarking/extracted-outputs/cell-25-3.svg" alt="Output of the previous code cell" />

<Image src="/docs/images/tutorials/refresh-backend-properties-with-real-time-benchmarking/extracted-outputs/cell-25-4.svg" alt="Output of the previous code cell" />

These plots require a careful reading, because not every gap between the two curves means the device has drifted:

- **Readout and single-qubit errors**: The measured values track the reported ones for most qubits, while the reported value for a handful of qubits is far off the measured curve. Those localized disagreements can reflect the drift this workflow is designed to catch.
- **$T_1$**: The two data sets scatter around each other with no systematic offset, which is consistent with $T_1$ fluctuating naturally over time.
- **Two-qubit errors**: The measured values sit systematically *above* the reported ones. This is expected because the reported values are measured on isolated gates, while the layer fidelity experiment runs many gates simultaneously and therefore includes crosstalk. The layered numbers are more relevant for deep circuits, but they are not directly comparable to the reported calibration data.
- **$T_2$**: The measured values sit well *below* the reported ones for most qubits. Again, methodology plays a large role, as examined in the next section.

#### Inspect individual experiment results

You can also inspect the raw results of the individual characterization experiments through the `experiment_data` property, which returns the underlying [Qiskit Experiments](https://qiskit-community.github.io/qiskit-experiments/) `ExperimentData` objects. For example, the following displays the measured $T_1$ decay curve of a single qubit, which is useful for checking the quality of a fit before trusting the number it produced.

In [12]:
t1_data = characterizer.experiment_data["t1"]

# Each qubit has its own fit figure
n_figs = len(t1_data.figure_names)
print(f"{n_figs} figures available, e.g. {t1_data.figure_names[0]}")

# Show the T1 decay curve of the first qubit
t1_data.figure(0)

156 figures available, e.g. T1_Q0_7aed29c2.svg


<Image src="/docs/images/tutorials/refresh-backend-properties-with-real-time-benchmarking/extracted-outputs/cell-28-1.avif" alt="Output of the previous code cell" />

#### Measure T2 with a dynamical decoupling train

By default the $T_2$ experiment applies a single Hahn echo and measures all qubits in parallel, which, as the comparison plot above showed, can yield noticeably lower $T_2$ values than the backend reports. One hypothesis is the number of echoes: the reported values may be calibrated with dynamical decoupling, which suppresses low-frequency noise. To test this, run a $T_2$-only characterization with `t2_num_echoes` set to a larger value, which applies a CPMG-style train of echo pulses. The delays represent the total free-evolution time in both cases, so the fitted $T_2$ values are directly comparable to the single-echo results.

In [13]:
# Run a T2-only characterization using a CPMG-style train of 8 echoes
characterizer_dd = BackendCharacterization(backend)
backend_updated_dd = characterizer_dd.run_and_update(
    experiments=["t2"], t2_num_echoes=8
)

Building T2 experiments


T2 (Hahn) experiment submitted: ['dalephg2fm4c73f1h4fg']


Updating T2


In [14]:
# Compare the multi-echo T2 against the backend-reported values
plot_characterization_comparison(
    old_props=backend.properties().to_dict(),
    new_props=backend_updated_dd.properties().to_dict(),
    plots=["t2"],
    title_prefix="8-echo CPMG",
)

# Compare medians across the reported values, the single-echo run,
# and the 8-echo run
reported_t2s = [
    prop["value"]
    for qubit in backend.properties().to_dict()["qubits"]
    for prop in qubit
    if prop["name"] == "T2"
]
t2_reported = np.median(reported_t2s)
t2_single = np.median(list(error_maps["t2_map"].values())) * 1e6
t2_dd = (
    np.median(list(characterizer_dd.analyze_results()["t2_map"].values()))
    * 1e6
)
print(
    f"Median T2 — reported: {t2_reported:.0f} us | "
    f"single echo: {t2_single:.0f} us | 8-echo CPMG: {t2_dd:.0f} us"
)

<Image src="/docs/images/tutorials/refresh-backend-properties-with-real-time-benchmarking/extracted-outputs/cell-31-0.svg" alt="Output of the previous code cell" />

Median T2 — reported: 138 us | single echo: 51 us | 8-echo CPMG: 56 us


In this run the extra echoes barely changed the result: the 8-echo median (56 &micro;s) is close to the single-echo one (51 &micro;s), and both remain far below the reported median (138 &micro;s). The number of echoes alone does not explain this gap; other methodological differences are involved, such as calibration procedure details, measuring all qubits in parallel rather than in isolation, and so forth.

The practical lesson is to treat $T_2$ (and layered two-qubit error) comparisons across methodologies with care. The refreshed values are a self-consistent snapshot taken under conditions close to a real workload, which makes them well suited for comparing qubits against each other or tracking a device over time. A gap relative to the reported calibration data, however, is not automatically evidence of drift.

### Steps 1–4 compressed into a single call

For day-to-day use you rarely need the intermediate results. The `run_and_update` method chains everything you did above (building, submitting, fitting, and updating) into a single call that returns the refreshed backend. It accepts the same `experiments` list (and forwards options such as `t2_num_echoes`), and can likewise be wrapped in a `Batch` or `Session` context.

<Admonition type="note">
The following cell does not return until all jobs and post-processing are done.
</Admonition>

In [15]:
# Initialize a fresh characterizer
characterizer = BackendCharacterization(backend)

# Run all of the experiments and update the backend in a single call
backend_updated = characterizer.run_and_update(experiments=experiments)

Building readout experiments


Building 1Q RB experiments


Building 2Q RB experiments


Building T1 experiments


Building T2 experiments


...

Updating single-qubit X error


Updating single-qubit SX error


Updating two-qubit error


Updating T1


Updating T2


The refreshed `backend_updated` object is now ready for the circuit comparison below. Refresh the properties close to execution time to reduce the effect of further device drift.

## Use refreshed properties for qubit selection

This optional example completes the workflow by comparing circuits compiled using reported properties with circuits compiled using refreshed properties. It uses the `backend` and `backend_updated` objects from above and submits additional QPU work.

### Build the comparison circuits

Prepare a chain of entangled qubits and measure its two endpoints. Ideally, the outcomes `00` and `11` each have probability 0.5. Vary the chain length to explore how circuit size affects the benefit of refreshed properties. The lengths below adapt to the selected backend; reduce the list to save QPU time.

In [1]:
ideal_dist = {"00": 0.5, "11": 0.5}
num_qubits_list = list(range(10, backend.num_qubits, 20)) + [
    backend.num_qubits
]
circuits = []
for num_qubits in num_qubits_list:
    circuit = QuantumCircuit(num_qubits, 2)
    circuit.h(0)
    for qubit in range(num_qubits - 1):
        circuit.cx(qubit, qubit + 1)
    circuit.barrier()
    circuit.measure([0, num_qubits - 1], [0, 1])
    circuits.append(circuit)

### Transpile with reported and refreshed properties

The transpiler uses a `Target` to choose physical qubits and native instructions. Updating backend properties alone can leave an already-built target unchanged. Copy the original target and apply the refreshed errors and coherence times explicitly, preserving instruction durations and connectivity. If a coherence time is unavailable for a qubit, keep its existing target value.

Use the same optimization level and random seed for both compilations so that the comparison controls for transpiler randomness.

In [2]:
target_updated = deepcopy(backend.target)
properties_updated = backend_updated.properties()

for gate in properties_updated.gates:
    if not any(
        parameter.name == "gate_error" for parameter in gate.parameters
    ):
        continue
    qargs = tuple(gate.qubits)
    if gate.gate not in target_updated.operation_names:
        continue
    if qargs not in target_updated[gate.gate]:
        continue
    instruction_properties = deepcopy(target_updated[gate.gate][qargs])
    if instruction_properties is not None:
        instruction_properties.error = properties_updated.gate_error(
            gate.gate, qargs
        )
        target_updated.update_instruction_properties(
            gate.gate, qargs, instruction_properties
        )

for qubit in range(backend.num_qubits):
    measurement_properties = deepcopy(target_updated["measure"][(qubit,)])
    measurement_properties.error = properties_updated.readout_error(qubit)
    target_updated.update_instruction_properties(
        "measure", (qubit,), measurement_properties
    )
    qubit_properties = properties_updated.qubit_property(qubit)
    if "T1" in qubit_properties:
        target_updated.qubit_properties[qubit].t1 = qubit_properties["T1"][0]
    if "T2" in qubit_properties:
        target_updated.qubit_properties[qubit].t2 = qubit_properties["T2"][0]

pm_reported = generate_preset_pass_manager(
    target=backend.target, optimization_level=3, seed_transpiler=42
)
pm_refreshed = generate_preset_pass_manager(
    target=target_updated, optimization_level=3, seed_transpiler=42
)
isa_reported = pm_reported.run(circuits)
isa_refreshed = pm_refreshed.run(circuits)

### Execute both sets of circuits

Run both versions on the same QPU, with the same shot count and suppression settings. Interleave reported and refreshed versions in the submission and repeat the comparison three times to observe variation. Job submission order does not guarantee execution order or eliminate drift.

In [3]:
n_trials = 3
shots = 4096
interleaved_circuits = [
    circuit for pair in zip(isa_reported, isa_refreshed) for circuit in pair
]
sampler = SamplerV2(mode=backend)
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
comparison_job = sampler.run(interleaved_circuits * n_trials, shots=shots)
print(f"Comparison job ID: {comparison_job.job_id()}")

Comparison job ID: dalfg5gpqrnc739633j0


### Compare the output distributions

Compute the Hellinger fidelity between each measured distribution and the ideal endpoint distribution. A value of 1 means the distributions match. This measures agreement in the computational basis; it does not establish quantum-state fidelity or certify entanglement.

The plot shows the mean across trials, with error bars indicating one standard deviation.

In [1]:
comparison_results = comparison_job.result()
fidelities = np.empty((n_trials, len(num_qubits_list), 2))
for trial in range(n_trials):
    for chain_index in range(len(num_qubits_list)):
        for variant in range(2):
            result_index = (
                trial * 2 * len(num_qubits_list) + 2 * chain_index + variant
            )
            counts = comparison_results[result_index].data.c.get_counts()
            fidelities[trial, chain_index, variant] = hellinger_fidelity(
                ideal_dist, counts
            )

fig, ax = plt.subplots(figsize=(8, 5))
for variant, label in enumerate(
    ["Reported properties", "Refreshed properties"]
):
    ax.errorbar(
        num_qubits_list,
        fidelities[:, :, variant].mean(axis=0),
        yerr=fidelities[:, :, variant].std(axis=0),
        fmt="o-",
        label=label,
    )
ax.set_xlabel("Chain length")
ax.set_ylabel("Hellinger fidelity of endpoint distribution")
ax.set_xticks(num_qubits_list, labels=num_qubits_list, rotation=45)
ax.legend()
ax.grid(True)
fig.tight_layout()
plt.show()

<Image src="/docs/images/tutorials/refresh-backend-properties-with-real-time-benchmarking/extracted-outputs/selection-results-0.svg" alt="Output of the previous code cell" />

In this run, refreshing increased the mean endpoint-distribution fidelity from 0.854 to 0.876 for the 10-qubit chain, but reduced it from 0.636 to 0.504 for the 30-qubit chain. Many longer-chain results are close to 0.5. A uniform distribution over the four possible outcomes also scores 0.5 against the ideal distribution, so a value near 0.5 alone is not evidence of useful endpoint correlations.

Refreshing properties does not guarantee better results. Both compilations may select the same qubits, and larger circuits leave less freedom to avoid poorly performing regions. Shot noise, device drift, and differences in characterization methodology also affect the comparison. Use the measured change and its variation to decide whether refreshing is useful for your workload.

## Next steps

If you found this work interesting, you might be interested in the following material:
<Admonition type="tip" title="Recommendations">
- Apply the qubit-selection comparison to your own circuits and evaluate whether the benefit justifies the characterization cost
- Learn how the reported calibration data is exposed in the [QPU information](/docs/guides/qpu-information) guide
- Explore the individual characterization experiments in the [Qiskit Experiments documentation](https://qiskit-community.github.io/qiskit-experiments/)
- Browse further device-level benchmarks in the [Qiskit Device Benchmarking repository](https://github.com/qiskit-community/qiskit-device-benchmarking)
</Admonition>